In [1]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import re
import json

In [2]:
url = "https://services.swpc.noaa.gov/json/rtsw/rtsw_wind_1m.json"
#url = "https://services.swpc.noaa.gov/json/rtsw/rtsw_mag_1m.json"
data = requests.get(url).json()
df = pd.DataFrame(data)

for i in range(20):
    print(data[i])
df
# for i in range(20):
#     print(data[i+1000])

{'time_tag': '2026-08-17T16:55:07', 'active': False, 'source': 'IMAP', 'proton_speed': 327.15, 'proton_temperature': 125802, 'proton_density': 13.25, 'proton_vx_gse': None, 'proton_vy_gse': None, 'proton_vz_gse': None, 'proton_vx_gsm': None, 'proton_vy_gsm': None, 'proton_vz_gsm': None, 'proton_sample_size': 1, 'alpha_speed': None, 'alpha_temperature': None, 'alpha_density': None, 'alpha_vx_gse': None, 'alpha_vy_gse': None, 'alpha_vz_gse': None, 'alpha_vx_gsm': None, 'alpha_vy_gsm': None, 'alpha_vz_gsm': None, 'alpha_sample_size': None, 'max_convergence_flag': 0, 'max_data_flag': 0, 'max_error_count_flag': 0, 'max_processing_flag': 0, 'max_range_flag': 0, 'max_sample_count_flag': 0, 'max_telemetry_flag': 0, 'overall_quality': 0}
{'time_tag': '2026-08-17T16:54:07', 'active': False, 'source': 'IMAP', 'proton_speed': 328.07, 'proton_temperature': 128806, 'proton_density': 13.98, 'proton_vx_gse': None, 'proton_vy_gse': None, 'proton_vz_gse': None, 'proton_vx_gsm': None, 'proton_vy_gsm': No

,time_tag,active,source,proton_speed,proton_temperature,proton_density,proton_vx_gse,proton_vy_gse,proton_vz_gse,proton_vx_gsm,...,alpha_vz_gsm,alpha_sample_size,max_convergence_flag,max_data_flag,max_error_count_flag,max_processing_flag,max_range_flag,max_sample_count_flag,max_telemetry_flag,overall_quality
0,2026-08-17T16:55:07,False,IMAP,327.15,125802.0,13.25,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
1,2026-08-17T16:54:07,False,IMAP,328.07,128806.0,13.98,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
2,2026-08-17T16:54:00,True,SOLAR1,336.60,117897.0,11.47,-336.0,-17.5,-10.5,-336.0,...,None,None,0,0,0,0,0,0,0,0
3,2026-08-17T16:54:00,False,ACE,359.31,90702.0,5.62,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
4,2026-08-17T16:53:07,False,IMAP,329.09,107300.0,14.08,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3590,2026-08-16T17:00:00,True,SOLAR1,299.90,24811.0,2.38,-298.6,1.8,-28.3,-298.6,...,None,None,0,0,0,0,0,0,0,0
3591,2026-08-16T17:00:00,False,ACE,303.20,15646.0,1.80,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
3592,2026-08-16T16:59:07,False,IMAP,307.02,16821.0,8.37,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
3593,2026-08-16T16:59:00,False,ACE,305.79,20649.0,2.00,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0


There seem to be two types of sources/satellites that NOAA pulls from: ACE and SOLAR1\. Only one of them is active at each timestamp\. That's the subset of data we will use for our real\-time dataset\.

In [3]:
df[df["active"]]

,time_tag,active,source,proton_speed,proton_temperature,proton_density,proton_vx_gse,proton_vy_gse,proton_vz_gse,proton_vx_gsm,...,alpha_vz_gsm,alpha_sample_size,max_convergence_flag,max_data_flag,max_error_count_flag,max_processing_flag,max_range_flag,max_sample_count_flag,max_telemetry_flag,overall_quality
2,2026-08-17T16:54:00,True,SOLAR1,336.6,117897.0,11.47,-336.0,-17.5,-10.5,-336.0,...,None,None,0,0,0,0,0,0,0,0
5,2026-08-17T16:53:00,True,SOLAR1,343.1,128548.0,12.47,-341.8,-13.8,-26.1,-341.8,...,None,None,0,0,0,0,0,0,0,0
8,2026-08-17T16:52:00,True,SOLAR1,343.2,127290.0,11.10,-342.0,-12.6,-26.3,-342.0,...,None,None,0,0,0,0,0,0,0,0
11,2026-08-17T16:51:00,True,SOLAR1,342.1,118722.0,13.66,-341.4,-19.1,-11.4,-341.4,...,None,None,0,0,0,0,0,0,0,0
14,2026-08-17T16:50:00,True,SOLAR1,343.2,116638.0,13.00,-342.5,-17.8,-12.9,-342.5,...,None,None,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3582,2026-08-16T17:03:00,True,SOLAR1,298.3,29408.0,2.52,-297.1,1.6,-27.1,-297.1,...,None,None,0,0,0,0,0,0,0,0
3585,2026-08-16T17:02:00,True,SOLAR1,298.5,28254.0,2.45,-297.4,1.8,-25.8,-297.4,...,None,None,0,0,0,0,0,0,0,0
3588,2026-08-16T17:01:00,True,SOLAR1,300.1,31424.0,2.36,-298.8,3.1,-26.9,-298.8,...,None,None,0,0,0,0,0,0,0,0
3590,2026-08-16T17:00:00,True,SOLAR1,299.9,24811.0,2.38,-298.6,1.8,-28.3,-298.6,...,None,None,0,0,0,0,0,0,0,0


In [4]:
active_df = df[df["active"]].copy()
active_df["time_tag"].is_unique

True

In [6]:
def get_mag_data():
    url = "https://services.swpc.noaa.gov/json/rtsw/rtsw_mag_1m.json"
    df = pd.DataFrame(requests.get(url).json())
    df["active"] = df["active"].astype(bool)
    df = (
        df[df["active"]]
        [["time_tag", "bt", "bx_gsm", "by_gsm", "bz_gsm"]]
    )
    return df

def get_proton_data():
    url = "https://services.swpc.noaa.gov/json/rtsw/rtsw_wind_1m.json"
    df = pd.DataFrame(requests.get(url).json())
    df["active"] = df["active"].astype(bool)
    df = (
        df[df["active"]]
        [["time_tag", "proton_speed", "proton_density"]]
    )
    return df


FEATURES = ["bt","bx_gsm","by_gsm","bz_gsm","proton_speed","proton_density"]
def get_real_time_data(interval=480):

    # Get separate datasets
    mag_df = get_mag_data()
    proton_df = get_proton_data()

    # Merge
    df = pd.merge(
        mag_df,
        proton_df,
        on="time_tag",
        how="outer"
    )

    # Convert datatypes
    df["time_tag"] = pd.to_datetime(df["time_tag"])
    for col in FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df[FEATURES] = df[FEATURES].replace(-9999, np.nan)

    # Get latest timestamp with no NaNs, go back 8 hours/interval from here
    df = df.sort_values("time_tag")
    df = df.drop_duplicates(subset="time_tag", keep="last")
    complete = df.dropna(subset=FEATURES)
    latest = complete["time_tag"].iloc[-1]

    
    full_index = pd.date_range(
        end=latest,
        periods=interval,
        freq="1min"
    )
    print(df["time_tag"].is_unique)
    df = (
        df.set_index("time_tag")
          .reindex(full_index)
          .rename_axis("time_tag")
          .reset_index()
    )

    df = df.rename(columns={
        "time_tag": "datetime",
        "bt": "mag_avg_nt",
        "bx_gsm": "bx_gsm_nt",
        "by_gsm": "by_gsm_nt",
        "bz_gsm": "bz_gsm_nt",
        "proton_speed": "flow_speed_km_s",
        "proton_density": "proton_density_n_cc"
    })

    return df

df = get_real_time_data()
df

True


,datetime,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,proton_density_n_cc
0,2026-08-17 08:56:00,5.81,-3.91,4.25,0.59,301.2,9.80
1,2026-08-17 08:57:00,5.84,-4.00,4.23,0.42,302.5,9.87
2,2026-08-17 08:58:00,6.41,-1.62,5.91,-1.82,297.8,9.39
3,2026-08-17 08:59:00,6.56,-1.20,6.22,-1.69,298.9,9.39
4,2026-08-17 09:00:00,6.51,-1.06,6.20,-1.67,300.1,9.16
...,...,...,...,...,...,...,...
475,2026-08-17 16:51:00,10.05,5.11,-8.06,2.38,342.1,13.66
476,2026-08-17 16:52:00,10.13,5.62,-7.34,3.79,343.2,11.10
477,2026-08-17 16:53:00,10.09,5.80,-6.16,5.35,343.1,12.47
478,2026-08-17 16:54:00,10.25,6.79,-6.79,3.56,336.6,11.47


In [7]:
df.isnull().sum()

datetime               0
mag_avg_nt             0
bx_gsm_nt              0
by_gsm_nt              0
bz_gsm_nt              0
flow_speed_km_s        9
proton_density_n_cc    9
dtype: int64

In [8]:
df.describe()

,datetime,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,proton_density_n_cc
count,480,480.000000,480.000000,480.000000,480.000000,471.000000,471.000000
mean,2026-08-17 12:55:30,8.801750,2.582958,0.959854,2.390375,321.830361,10.946815
min,2026-08-17 08:56:00,5.580000,-4.930000,-9.920000,-6.950000,295.500000,6.760000
25%,2026-08-17 10:55:45,7.497500,-1.872500,-3.970000,-0.675000,303.400000,9.790000
50%,2026-08-17 12:55:30,8.135000,3.960000,0.475000,1.940000,319.700000,10.380000
75%,2026-08-17 14:55:15,10.235000,6.795000,6.480000,6.045000,339.550000,11.215000
max,2026-08-17 16:55:00,12.610000,9.530000,7.830000,10.130000,353.000000,19.310000
std,NaN,1.826331,4.541457,5.307385,4.284249,17.709199,2.299929


## Integrating with Pipeline Thus Far

In [1]:
from scripts import pipeline as Pipeline

In [2]:
# Can use "0hr", "3hr", or "6hr"
# You can also set "verbose=True" and/or "benchmark=True" if you would like verbose output
# or to time each step in the pipeline
pred_kp, pred_time = Pipeline.RT_Pipeline("0hr", verbose=True, benchmark=True)
print(f"Predicted KP {pred_kp} at timestamp (UTC) {pred_time}")

Fetching real-time data took 0.564491 seconds
mag_avg_nt: 0 values interpolated
bx_gsm_nt: 0 values interpolated
by_gsm_nt: 0 values interpolated
bz_gsm_nt: 0 values interpolated
flow_speed_km_s: 9 values interpolated
proton_density_n_cc: 9 values interpolated
--------------------------------------------------
Final NaN cleaning results:
Rows before: 480
Rows after:  480
Rows removed: 0
Percent removed: 0.00%
Total values interpolated: 18
--------------------------------------------------
Cleaning NaNs took 0.002504 seconds
Adding time columns took 0.001801 seconds
Added mag_avg_nt_log — min=1.8840, max=2.6108, NaNs=0
Added flow_speed_km_s_log — min=5.6920, max=5.8693, NaNs=0
Added proton_density_n_cc_log — min=2.0490, max=3.0111, NaNs=0
Log transform took 0.001756 seconds
Found 0 rows with duplicate (year, day, hour, minute) combinations
--- Record count audit ---
Input rows (per-minute): 480
Resampled hourly buckets (2026-08-17 09:00:00 to 2026-08-17 16:00:00): 8
Candidate 3-hour int

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>